# Round-trip (back-translation) augmentation - StereoQueerEval Task A

Produces paraphrases of the **train split** by translating each row into a pivot
language and back (`IT->EN->IT`, `NL->EN->NL`, `EN->DE->EN`), keeping the gold
labels. The outputs (`aug_rt_*_training.tsv`) plug straight into
`train.py --train-extras`.

Why it is the safest MT augmentation: same language, labels preserved, and
generation is restricted to the train split (same `--split-seed` / `--val-fraction`
as training), so a paraphrase of a held-out video can never leak into training.

The thing to worry about is **semantic drift** (label noise): round-tripping a
`label=1` row can smooth sarcasm/slurs into a neutral sentence that is still
tagged `yes`. So this notebook:

1. runs a **random probe** (not the full set),
2. scores drift with multilingual embeddings and reports a **95% Wilson CI**,
3. dumps a small pair table so you can eyeball label-1 retention,
4. only then runs the full augmentation with light sanity thresholds.

A row counts as drifted if its round-trip/source embedding cosine is below
`--threshold` (default 0.8) or if a hand label in the review TSV says so.


## 1. Setup

Clones/updates `src/` and installs requirements. `sentencepiece` is required by
the NLLB tokenizer (pinned in `requirements.txt`; re-asserted here). The model is
downloaded on first use -- a GPU (Kaggle T4) turns the full run from hours into
minutes.


In [ ]:
import os
import subprocess, sys

if not os.path.isdir('Try'):
    subprocess.run('git clone --depth 1 https://github.com/AmnO-O/Try.git', shell=True, check=True)
else:
    subprocess.run('git -C Try pull --ff-only', shell=True, check=True)
os.chdir('Try')
os.makedirs('data', exist_ok=True)
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
subprocess.run('pip install -q sentencepiece', shell=True, check=True)

import torch
print('kernel :', 'Kaggle' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'Colab' if 'COLAB_GPU' in os.environ else 'local')
print('cuda   :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo   :', os.getcwd())
!ls src


## 1. Locate the TSVs

Downloads are gitignored, so the clone has no data. This searches the usual
places -- Kaggle dataset path, Colab `/content`, local `data/` and `LGBT/` -- and
sets `LGBT_DATA_DIR` to the directory that actually holds the files.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.environ['KAGGLE_INPUT_PATH'])
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return os.path.dirname(hits[0]), hits
    return None, []

data_dir, found = _first(candidates)
if not found:
    raise RuntimeError('No StereoQueerEval_*_training.tsv found. Add your Kaggle dataset and re-run.')
os.environ['LGBT_DATA_DIR'] = data_dir
print('data dir :', data_dir)
print('files    :', [os.path.basename(p) for p in found])
if any('_FA_' in os.path.basename(p) for p in found):
    print('WARNING: a Persian training file is present; it is eval-only and is not used here.')
assert {'EN', 'IT', 'NL'} <= {os.path.basename(p).split('_')[1] for p in found}, found


## 2. Sanity check the data

Row counts and label balance via the same loader the augmentation uses.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import src.data as data

df = data.load_frame()
print('rows per language :', df['lang'].value_counts().to_dict())
print('labels            :', df['stereotype'].value_counts().to_dict())
print('total             :', len(df))


## 3. Random probe (fast)

Round-trips a **random sample**, not the full set. `--fields comment` is enough
to judge drift of the labelled text, and `--review-n 60` yields ~180 pairs total
-- a comfortable sample for the CI below. On a T4 this is a couple of minutes.


In [ ]:
import subprocess

def run(cmd, check=True):
    print('$', cmd, '\n')
    return subprocess.run(cmd, shell=True, check=check)

run('python src/augment_roundtrip.py --languages EN IT NL --limit 60 --fields comment '
    '--split-seed 42 --review-n 60 --review-label all --qc-samples 5 --qc-label yes --keep-emoji --num-beams 4 --min-cosine 0.8 '
    '--out-prefix aug_rt_probe')


## 4. Drift estimate with a confidence interval

Embeds source vs round-trip comments with a multilingual sentence encoder and
reports the drift rate with a **95% Wilson interval**, plus mean cosine and token
overlap. If 0.8 flags too much informal/sarcastic text, lower `--threshold`.

Reported drift is a proxy: it says *how much meaning moved*, not whether the
stereotype specifically survived. That is what the next cell is for.


In [ ]:
import glob

reviews = sorted(glob.glob('aug_rt_probe_review_*.tsv'))
print('review files:', reviews)
run('python src/eval_drift.py ' + ' '.join(reviews) + ' --threshold 0.8')


## 4. Eyeball the pairs (label-1 first)

Read ~15 label-1 pairs and ask: does the stereotype still read as one, or has it
been smoothed into a neutral sentence? Sort puts the most eroded rows first
(`token_overlap` ascending), which is where drift will show up.


In [ ]:
import glob
import pandas as pd

pd.set_option('display.max_colwidth', 400)
pairs = pd.concat([pd.read_csv(p, sep='\t') for p in sorted(glob.glob('aug_rt_probe_review_*.tsv'))],
                  ignore_index=True)
pairs[['token_overlap', 'len_ratio', 'cosine']] = pairs[['token_overlap', 'len_ratio', 'cosine']].apply(pd.to_numeric, errors='coerce')
pairs = pairs.sort_values(['stereotype', 'cosine'], ascending=[False, True])
print(pairs.groupby('stereotype')['cosine'].agg(['count', 'mean', 'min']))
pairs[pairs['stereotype'] == 'yes'][['lang', 'cosine', 'token_overlap', 'len_ratio', 'yt_comment', 'roundtrip']].head(15)


## 5. Full augmentation (apply the thresholds)

Once the probe is acceptable, augment the whole train split with light filters:
`--min-cosine 0.8` is the semantic gate: it drops MT collapses and subtle meaning
smoothing (low embedding cosine) while keeping synonym / slur-swap paraphrases that a
lexical overlap test would wrongly reject. `--max-len-ratio 2.0` drops wild length
most eroded stereotypes are dropped. Re-check `aug_rt_review_*.tsv` afterwards.


In [ ]:
run('python src/augment_roundtrip.py --languages EN IT NL --fields all --split-seed 42 '
    '--min-cosine 0.8 --max-len-ratio 2.0 --keep-emoji --num-beams 4 '
    '--review-n 40 --review-label yes --out-prefix aug_rt')
!ls -la aug_rt_*_training.tsv


## 5. Train with the augmented rows

Feed the files via `--train-extras` and keep `--split-seed` identical to the
generation run so the extras align with the same split. Compare against the same
command **without** `--train-extras`.


In [ ]:
extras = ','.join(f'aug_rt_{lang}_to_{lang}_training.tsv' for lang in ('EN', 'IT', 'NL'))
print('extras:', extras)
run('python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 '
    '--readout-init random --warmup-batch-size 8 --warmup-epochs 2 --languages EN IT NL '
    '--split-seed 42 --seed 42 --out weights/prompt_rt_s42 --train-extras ' + extras)


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt && pip install sentencepiece
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/

# random probe + drift CI + pair table
python src/augment_roundtrip.py --languages EN IT NL --limit 60 --fields comment \
    --split-seed 42 --review-n 60 --keep-emoji --num-beams 4 --min-cosine 0.8 --out-prefix aug_rt_probe
python src/eval_drift.py aug_rt_probe_review_EN.tsv aug_rt_probe_review_IT.tsv aug_rt_probe_review_NL.tsv

# full augmentation with sanity filters
python src/augment_roundtrip.py --languages EN IT NL --fields all --split-seed 42 \
    --min-cosine 0.8 --max-len-ratio 2.0 --keep-emoji --num-beams 4 \
    --review-n 40 --review-label yes

# train on top of the augmented rows
python src/train.py --epochs 10 --batch-size 32 --max-len 320 --unfreeze-layers 2 \
    --readout-init random --warmup-batch-size 8 --warmup-epochs 2 --languages EN IT NL \
    --split-seed 42 --seed 42 --train-extras \
    aug_rt_EN_to_EN_training.tsv,aug_rt_IT_to_IT_training.tsv,aug_rt_NL_to_NL_training.tsv
```

Hand-annotate drift for a stricter CI: add a `drift` column (1/0) to a copy of a
review TSV and pass `--drift-label drift` to `src/eval_drift.py`.
